# 数据集蒸馏

简单解释一下为什么我将第二个专题选定为数据集蒸馏：我认为引入更多的工程技巧是永无止境的事情，但是跳脱到另外一个领域却可以在有限的步骤内为我们提供一种别样的思维方式。当我们学习了众多从建模损失函数与改动监督学习样本入手的方案之后，我们也应该思考如何从数据集本身入手。

关于数据集蒸馏，我需要指出他并不是图像生成的一个子领域，甚至不是生成式模型的一个子领域。这个领域的方法对于整个机器学习是泛用的。那么为什么我们要在一本图像生成教程中用一个专题来介绍？因为最当下的数据集蒸馏领域与 Diffusion 范式等等密不可分。

总之请让我们迅速开始。由于需要从零开始讲述一个领域，本章围绕几篇开山之作展开。推荐你读 https://arxiv.org/pdf/1811.10959 Dataset Distillation。

# 基本逻辑

数据集蒸馏核心任务非常简单，我们希望将数据集缩小，使得模型在我们的精简版特制数据集上训练成果接近原始数据集。

这种蒸馏的威力比想象中更大。对于极其简单的 MNIST 手写识别任务，我们可以将数据集蒸馏到每类仅仅一张图片，总共十张。在这十张图片上训练的模型，最终居然可以达到 $94\%$ 的识别正确率。

对于一般的训练，我们记数据集为 $x$，模型参数为 $\theta$，那么核心任务是
$$\theta^\star =\arg\min_\theta \ell(x,\theta)$$
其中 $\ell(x,\theta)$ 指模型在数据集上损失函数。我们希望找到一个小得多的合成数据集 $\tilde{x}$，满足
$$|\tilde{x}| \ll |x|$$
在合成数据集 $\tilde{x}$ 上，我们训练模型，希望其在真实验证集上水平接近在真实数据集上训练模型的表现。

下面这张图展示了蒸馏的结果。需要注意，合成数据集中每张图可以看起来完全不像真实图片。更多的，我们固定初始化是指对于一个固定的初始化模型特制了蒸馏数据集。

<img src="./assets/datadis.png" width="800" height="300">

对于标准训练和学习过程，在时间步 $t$，假设采样了数据 $\mathbf{x}_t= \{x_{t,j}\}_{j=1}^n$，那么梯度下降更新公式就是
$$\theta_{t+1} =
\theta_t - \eta \nabla_{\theta_t}\ell(\mathbf{x}_t,\theta_t)$$
此处的 $\eta$ 就是学习率。一般的学习过程需要非常非常多步数更新，这是我们不愿意看到的。

我们希望做一个蒸馏的合成数据集 $\tilde{\mathbf{x}} = \{\tilde{x}_i \}_{i=1}^M, M \ll N$ 和一个对应的学习率 $\tilde{\eta}$。那么一步更新就是
$$\theta_1 =
\theta_0 - \tilde{\eta}\nabla_{\theta_0}\ell(\tilde{\mathbf{x}},\theta_0)$$

这个蒸馏数据集和学习率满足
$$\tilde{\mathbf{x}}^*, \tilde{\eta}^* = \arg\min_{\tilde{\mathbf{x}}, \tilde{\eta}} \mathcal{L}(\tilde{\mathbf{x}}, \tilde{\eta}; \theta_0) := \arg\min_{\tilde{\mathbf{x}}, \tilde{\eta}} \ell(\mathbf{x}, \theta_1) = \arg\min_{\tilde{\mathbf{x}}, \tilde{\eta}} \ell(\mathbf{x}, \theta_0 - \tilde{\eta} \nabla_{\theta_0} \ell(\tilde{\mathbf{x}}, \theta_0))$$

其中
$$\mathcal{L}(\tilde{\mathbf{x}}, \tilde{\eta}; \theta_0) = \ell(\mathbf{x}, \theta_1)$$
我们详细解释一下上面这个公式。$\mathcal{L}$ 是我们定义的最小化目标，其定义就是我们对于初始权重 $\theta_0$ 优化一步之后的参数 $\theta_1$ 在真实数据集上损失。我们希望这个损失最小，换言之，$\tilde{\mathbf{x}}$ 产生的梯度方向有用。

请注意，$\mathcal{L}$ 是可微分的，这意味着我们可以使用梯度下降来优化这一目标。

以上是固定初始化的优化。但是对于随机初始化怎么办？实际上一个固定初始化得到的蒸馏数据集对于随机初始化是不具备泛化性的。因此我们需要调整优化目标。假设我们的初始化 $\theta_0$ 遵从分布 $p(\theta)$，那么我们需要
$$\tilde{x}^\star,\tilde{\eta}^\star =
\arg\min_{\tilde{x},\tilde{\eta}}
\mathbb{E}_{\theta_0\sim p(\theta_0)}
\mathcal{L}(\tilde{x},\tilde{\eta};\theta_0)$$
值得一提，随机初始化蒸馏得到的数据集看起来比固定初始化有理得多，简而言之就是数字看起来更像数字了。

<img src="./assets/datadisran.png" width="800" height="230">

随机初始化训练效果会略差一些，MNIST 数据集上正确率下降到约 $80\%$，这可以视为对于泛化性的牺牲。

### 蒸馏下界

现在我们来做一些对于直觉的推导。我们试图给出一个答案，回答蒸馏数据集到底可以做到多小尺寸。


假设我们的数据集是成对的集合 $\mathbf{x} = \{(d_i, t_i)\}_{i=1}^N$ ，这意味着可以写成矩阵 $d \in \mathbb{R}^{N\times D}$ 和目标值矩阵 $t \in \mathbb{R}^{N\times 1}$。符号很容易理解，比如 MNIST 中一张图片就对应了一个数字。

那么我们可以将损失函数 $\ell (\mathbf{x},\theta)$ 展开
$$\ell (\mathbf{x},\theta) = \ell ((d,t),\theta) = \frac{1}{2N}\|d\theta -t\|^2$$
这里的 $d\theta$ 直觉理解是矩阵乘法，实际上就是将 $d$ 输入神经网络的输出简写。

我们考虑对这个损失函数做梯度下降寻找最优解 $\theta^*$
$$\nabla_\theta \ell(\mathbf{x}, \theta) = \frac{1}{2N} \cdot 2\mathbf{d}^T(\mathbf{d}\theta - \mathbf{t}) = \frac{1}{N}(\mathbf{d}^T\mathbf{d}\theta - \mathbf{d}^T\mathbf{t})$$
对于最优解 $\theta^*$，一个必要条件就是极值点梯度为零，因此任意方向方向导数也为零。我们得到
$$\frac{1}{N}(\mathbf{d}^T\mathbf{d}\theta^* - \mathbf{d}^T\mathbf{t}) = \mathbf{0}$$
最后化简
$$\mathbf{d}^T\mathbf{d}\theta^* = \mathbf{d}^T\mathbf{t}$$
这是一个基本的机器学习结论。以上方程指出了 MSE 损失函数极值点必须满足的条件。

回到数据集蒸馏。现在我们蒸馏数据集则是 $\tilde{\mathbf{x}} = (\tilde{d}, \tilde{t})$，有 $\tilde{d}\in \mathbb{R}^{M\times D}, \tilde{t}\in\mathbb{R}^{M\times 1}, M \ll N$。

我们需要最小化 $\mathcal{L}(\tilde{\mathbf{x}}, \tilde{\eta}; \theta_0) = \ell(\mathbf{x}, \theta_1)$。

我们将参数更新过程写开
$$\theta_1 = \theta_0 - \tilde{\eta}\nabla_{\theta_0} \ell(\tilde{\mathbf{x}},\theta_0)$$
此处
$$\nabla_{\theta_0} \ell(\tilde{\mathbf{x}},\theta_0) = \frac{1}{M} \tilde{d}^T (\tilde{d}\theta_0 - \tilde{t})$$
所以更新之后就是
$$\theta_1 = \theta_0 - \tilde{\eta} \nabla_{\theta_0} \ell(\tilde{\mathbf{x}},\theta_0)
= \theta_0 - \frac{\tilde{\eta}}{M} \tilde{d}^T (\tilde{d}\theta_0 - \tilde{t})
= (I - \frac{\tilde{\eta}}{M} \tilde{d}^T \tilde{d}) \theta_0 + \frac{\tilde{\eta}}{M} \tilde{d}^T \tilde{t}$$

在极度理想情况下，我们希望 $\theta_1$ 就是最优解 $\theta^*$，这意味着我们可以使用刚刚推导的最优解结论。代入得到
$$\mathbf{d}^T\mathbf{d}(\mathbf{I} - \frac{\tilde{\eta}}{M}\tilde{\mathbf{d}}^T\tilde{\mathbf{d}})\theta_0 + \frac{\tilde{\eta}}{M}\mathbf{d}^T\mathbf{d}\tilde{\mathbf{d}}^T\tilde{\mathbf{t}} = \mathbf{d}^T\mathbf{t}$$

如果我们要求，对于随机初始化的 $\theta_0$ 都能有上式成立，实际上我们就要求了上式中 $\theta_0$ 系数直接为零矩阵，这意味着
$$\mathbf{I} - \frac{\tilde{\eta}}{M}\tilde{\mathbf{d}}^T\tilde{\mathbf{d}} = \mathbf{0}$$

单位矩阵 $\mathbf{I}$ 的是 $D \times D$ 维度秩为 $D$ 矩阵，这意味着$\tilde{\mathbf{d}}^T\tilde{\mathbf{d}}$ 也必须是为 $D$的。合成数据矩阵 $\tilde{\mathbf{d}}$ 的维度是 $M \times D$。根据矩阵乘法的秩不等式 $\text{rank}(\tilde{\mathbf{d}}^T\tilde{\mathbf{d}}) \le \text{rank}(\tilde{\mathbf{d}}) \le \min(M, D)$。为了使其秩达到 $D$，必然推导出
$$M \ge D$$

这是非常关键的直觉：合成数据的样本数 $M$ 必须大于或等于数据的特征维度 $D$。在真实的计算机视觉任务中，一张普通的图像特征维度 $D$ 也非常庞大。这意味着试图用极少量的图像去兼容模型所有的随机初始化路径在数学上是不可能的。

这意味着我们需要放弃一些假设。首先我们假定一步梯度下降结果 $\theta_1$ 就是最优参数 $\theta^*$，这个假设非常严苛，我们可以放松到多步梯度下降结果。更多的，我们放弃任意初始化假设。我们不要求合成数据适应所有的 $\theta_0$，而是将优化范围约束在特定的初始化分布 $p(\theta_0)$ 内，这可以大大减少我们理论上的负担。

### 多步梯度下降

一步梯度下降限制了训练的深入程度。现在我们给出多步梯度下降形式
$$\theta_{i+1} = \theta_i - \tilde{\eta}_i \nabla_{\theta_i} \ell(\tilde{\mathbf{x}}_i, \theta_i)$$
其实就是多优化几步。最终我们讨论损失 $\ell(\mathbf{x},\theta_{final})$。

更多的，我们对同一组蒸馏数据进行多轮训练，即循环使用蒸馏数据多次。每轮可以重新设置学习率，因为通常后期训练需要更小步长。这样能让少量蒸馏数据在多次梯度更新中模拟真实训练过程。换言之，训练中设置更多的 epoch。

一个关键的技巧是 Back-Gradient Optimization，这是一项计算优化技术。关于为什么需要使用这项技术，我们需要先介绍完整的数据集蒸馏算法。

# 蒸馏算法

我们先说单步算法，之后可以自然地延伸到多步。

首先我们给定模型初始权重分布 $p(\theta_0)$，数据集初始化 $\tilde{x}=\{\tilde{x}_i\}_{i=1}^{M}$，更新步长初始化 $\tilde{\eta}\leftarrow \tilde{\eta}_0$，每次取出真实数据集合 Batch Size $n$ 与总共优化蒸馏数据轮次 $T$。

对于每次迭代步数 $t \ge 1$，我们取出 mini-batch $x_t=\{x_{t,j}\}_{j=1}^{n}$ 和来自初始权重分布的多个初始化参数 $\theta_0^{(j)}\sim p(\theta_0)$。

对每个采样到的 $\theta_0^{(j)}$ 执行以下两步。

首先使用蒸馏数据集更新一次步长
$$\theta_1^{(j)}
=
\theta_0^{(j)}
-
\tilde{\eta}
\nabla_{\theta_0^{(j)}}
\ell\left(\tilde{x},\theta_0^{(j)}\right)$$

其次对于取出的 mini-batch 计算损失函数

$$\mathcal{L}^{(j)}
=
\ell\left(x_t,\theta_1^{(j)}\right)$$

对于所有采样的随机初始化权重做完上述两件事之后，我们计算平均损失
$$\sum_j \mathcal{L}^{(j)}$$
现在我们可以更新蒸馏数据集和学习率步长
$$\tilde{x}
\leftarrow
\tilde{x}
-
\alpha
\nabla_{\tilde{x}}
\sum_j \mathcal{L}^{(j)}$$
$$\tilde{\eta}
\leftarrow
\tilde{\eta}
-
\alpha
\nabla_{\tilde{\eta}}
\sum_j \mathcal{L}^{(j)}$$
请注意这里的 $\alpha$ 是更新数据集和学习率所使用的学习率，需要与模型更新的学习率做区分。

对以上过程重复直到迭代轮次 $t=T$。

以下是完整的算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Dataset Distillation} \\
\hline
\textbf{Input: } p(\theta_0)\text{: distribution of initial weights; } M\text{: the number of distilled data} \\
\textbf{Input: } \alpha\text{: step size; } n\text{: batch size; } T\text{: the number of optimization iterations; } \tilde{\eta}_0\text{: initial value for } \tilde{\eta} \\
\begin{aligned}
1: & \quad \text{Initialize } \tilde{\mathbf{x}} = \{\tilde{x}_i\}_{i=1}^M \text{ randomly, } \tilde{\eta} \leftarrow \tilde{\eta}_0 \\
2: & \quad \textbf{for each } \text{training step } t = 1 \text{ to } T \textbf{ do} \\
3: & \qquad \text{Get a minibatch of real training data } \mathbf{x}_t = \{x_{t,j}\}_{j=1}^n \\
4: & \qquad \text{Sample a batch of initial weights } \theta_0^{(j)} \sim p(\theta_0) \\
5: & \qquad \textbf{for each } \text{sampled } \theta_0^{(j)} \textbf{ do} \\
6: & \qquad\quad \text{Compute updated parameter with GD: } \theta_1^{(j)} = \theta_0^{(j)} - \tilde{\eta} \nabla_{\theta_0^{(j)}} \ell(\tilde{\mathbf{x}}, \theta_0^{(j)}) \\
7: & \qquad\quad \text{Evaluate the objective function on real training data: } \mathcal{L}^{(j)} = \ell(\mathbf{x}_t, \theta_1^{(j)}) \\
8: & \qquad \textbf{end for} \\
9: & \qquad \text{Update } \tilde{\mathbf{x}} \leftarrow \tilde{\mathbf{x}} - \alpha \nabla_{\tilde{\mathbf{x}}} \sum_j \mathcal{L}^{(j)}, \text{ and } \tilde{\eta} \leftarrow \tilde{\eta} - \alpha \nabla_{\tilde{\eta}} \sum_j \mathcal{L}^{(j)} \\
10:& \quad \textbf{end for}
\end{aligned} \\
\textbf{Output: } \text{distilled data } \tilde{\mathbf{x}} \text{ and optimized learning rate } \tilde{\eta} \\
\hline
\end{array}$$

请注意这里的梯度更新，这里的梯度计算是合法的因为整个链路是可微的
$$\tilde{x}
\longrightarrow
\ell(\tilde{x},\theta_0)
\longrightarrow
\nabla_{\theta_0}\ell(\tilde{x},\theta_0)
\longrightarrow
\theta_1
\longrightarrow
\ell(x_t,\theta_1)$$
但是也没有那么简单。因为我们这里对一个梯度求了梯度，这是一个二阶的行为。展开来说，首先我们更新模型参数取出了一次梯度
$$\theta_1
=
\theta_0
-
\tilde{\eta}
\nabla_{\theta_0}\ell(\tilde{x},\theta_0)$$

其次我们对于损失本身又求了一次梯度，这导致我们对梯度求了梯度，换言之
$$\frac{\partial \theta_1}{\partial \tilde{x}}
=
-
\tilde{\eta}
\frac{\partial}{\partial \tilde{x}}
\left[
\nabla_{\theta_0}
\ell(\tilde{x},\theta_0)
\right]$$
此处的计算肉眼可见很昂贵。

关于多步梯度下降版本，实际上就是将单步替换。假设我们设定多步更新步数 $K$，对于一步更新我们原本只需要准备一个 蒸馏数据集，现在需要准备 $K$ 个蒸馏数据集 $\tilde{x}_0,\tilde{x}_1,\dots,\tilde{x}_{K-1}$ 与学习率 $\tilde{\eta}_0,\tilde{\eta}_1,\dots,\tilde{\eta}_{K-1}$。

从 $\theta_0$ 出发更新 $K$ 次
$$\theta_{i+1} =
\theta_i -
\tilde{\eta}_i
\nabla_{\theta_i}
\ell(\tilde{x}_i,\theta_i),
\qquad i=0,\dots,K-1$$

我们对于最终的参数求在 mini-batch 上损失函数
$$\mathcal{L}
=
\ell(x,\theta_K)$$
所以内层本质在优化
$$\min_{\tilde{x}_0,\dots,\tilde{x}_{K-1},
\tilde{\eta}_0,\dots,\tilde{\eta}_{K-1}}
\mathbb{E}_{\theta_0\sim p(\theta_0)}
\left[
\ell(x,\theta_K)
\right]$$
之后的更新数据集和学习率步长是完全类似的，只是我们需要对 $K$ 个数据集和学习率都要更新一遍。

同样肉眼可见，多步更新时整体计算成本会昂贵得多得多。因为每个 $\theta_{i+1}$ 更新都依赖 $\theta_i$ 和前面的所有蒸馏数据集
$$\tilde{x}_0
\to
\theta_1
\to
\tilde{x}_1
\to
\theta_2
\to
\cdots
\to
\theta_K$$
这意味着反向传播会穿透整个链路。

原始朴素反向传播中，我们需要保存之前产生的所有梯度图和参数，这会对显存造成巨大的压力。

这里我们重提这个非常重要的优化技巧，Back-Gradient Optimization。实际上这个优化技巧又被称为 Hessian-vector product，非常类似 Jacobian-vector product 的思想。当我们准备让 Hessian 矩阵与某个向量做乘积时，我们不需要在显存中保留完整的 Hessian 矩阵而是直接保留其与目标向量的乘积，这个乘积的储存成本远低于原始 Hessian 矩阵。

我向你展示如何使用。

我们希望对 $\mathcal{L} = \ell(x, \theta_K)$ 关于每个 $\tilde{x}_i$ 和 $\tilde{\eta}_i$ 求梯度。

首先定义
$$v_i = \frac{\partial \mathcal{L}}{\partial \theta_i}$$
那么存在递推关系
$$v_i = \frac{\partial \theta_{i+1}}{\partial \theta_i}^T \frac{\partial \mathcal{L}}{\partial \theta_{i+1}} = \frac{\partial \theta_{i+1}}{\partial \theta_i}^T v_{i+1}$$

现在我们演示 $\frac{\partial \theta_{i+1}}{\partial \theta_i}$ 的计算。由于关系
$$\theta_{i+1} = \theta_i - \tilde{\eta}_i \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i)$$
可以得到
$$\frac{\partial \theta_{i+1}}{\partial \theta_i} = I - \tilde{\eta}_i \frac{\partial}{\partial \theta_i} \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i) = I - \tilde{\eta}_i H_i$$
其中 $H_i = \nabla_{\theta_i}^2 \ell(\tilde{x}_i, \theta_i)$ 就是 Hessian 矩阵。

因此我们可以得到关系
$$v_i = \left(\frac{\partial \theta_{i+1}}{\partial \theta_i}\right)^T v_{i+1} = (I - \tilde{\eta}_i H_i)^T v_{i+1}$$
根据最终外层梯度
$$v_K =
\nabla_{\theta_K}
\ell(x,\theta_K)$$
我们利用递推关系得到每个 $v_i$。

但是 $v_i$ 仅仅是一个中间量。关于最终如何求出对于蒸馏数据集的梯度，我们利用一个关键关系
$$\nabla_{\tilde{x}_i} \mathcal{L} = \frac{\partial \theta_{i+1}}{\partial \tilde{x}_i}^T v_{i+1}$$
这使得我们可以得到
$$\nabla_{\tilde{x}_i} \mathcal{L} = -\tilde{\eta}_i (\partial_{\tilde{x}_i} \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i))^T v_{i+1}$$
请注意这里的 $\partial_{\tilde{x}_i} \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i)$，这个量是可以独立求出的，换言之这个量和 $\theta_{j}, \tilde{x}_j, j \neq i$ 完全没有关系。
更多的，我们给出学习率关于损失的梯度写法
$${
\nabla_{\tilde{\eta}_i} \mathcal{L} = - v_{i+1}^T \, \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i)
}$$

至此我们得到了所有相关量的计算方法。但是我们提到的 Back-Gradient Optimization 技术被应用在哪里？实际上是项
$$(\partial_{\tilde{x}_i} \nabla_{\theta_i} \ell(\tilde{x}_i, \theta_i))^T v_{i+1}$$
这是一个 Hessian-vector product，换言之 Hessian 矩阵乘上一个向量。优化技巧也非常简单
$$H v = (\partial_{\tilde{x}_i} \nabla_\theta \ell) ^T v = \partial_{\tilde{x}_i} ((\nabla_\theta \ell)^T \cdot \text{sg}(v))$$
请注意这里的停训算子 $\text{sg}$，这非常重要，因为我们需要在求梯度过程中将 $v$ 视为一个常量，这才能拉入括号提前计算。如果没有这个停训算子，我们直接将 $v$ 拉入括号会导致间接地对于其求一次梯度，这和在括号外面相乘是不等价的。

以上这个技巧保证我们没有任何一个时间储存完整的 Hessian 矩阵，大大减少计算成本。

更多的，计算 $v_i$ 实际上也需要反向传播优化这个技巧。具体而言
$$v_i = (I - \tilde{\eta}_i H_i)^T v_{i+1}$$
其中 $H_i = \nabla_{\theta_i}^2 \ell(\tilde{x}_i, \theta_i)$。我们使用同样的技巧
$$H v = \nabla_\theta ((\nabla_\theta \ell)^T \cdot \text{sg}(v))$$


# 恶意更新

最后我们简短说一说作者讨论的蒸馏数据集投毒问题，这还比较有趣。简而言之就是我们把某个标签 $K$ 改为 $T$，蒸馏出的数据集就会鼓励模型将 $K$ 类别识别为 $T$。换言之优化
$$\tilde{x}^{*},\tilde{\eta}^{*}
=
\arg\min_{\tilde{x},\tilde{\eta}}
\mathbb{E}_{\theta_0\sim p(\theta_0)}
\mathcal{L}_{K\to T}
(\tilde{x},\tilde{\eta};\theta_0)$$
听起来这个结论比较频繁。实际上作者想要表达，蒸馏数据集被投毒危害大于一般数据集被投毒，因为样本更少了。

# 总结

本章我们讲述了数据集蒸馏的开山之作，这篇文章首次将对于数据集的优化拉入视野。

但是作为早期作品问题也很明显，那就是 Bilevel Optimization 带来的计算负担。即使在 HVP 技术帮助下，这还是难以在大规模数据集上开展。

下一章为你带来 Gradient Matching 和 Distribution Matching，我们提出一些进阶想法。